# 08 Validate: every test from the test plan

**What this notebook does.** I wrote down every test before running anything: T1 to T10
check the requirements, V1 to V4 check that the work itself is correct, QC-01 to QC-09 check the quality criteria.
This notebook carries out exactly those tests, one per cell, and prints the results as three tables: validation,
verification and quality criteria.

**Validation and verification.** Validation asks: does it do what the requirements say? Verification asks: does it
work correctly? Both are tested by a standard procedure. That's why this is a
separate notebook: every check runs from the saved files, the same way every time.

**It doesn't open the test part again.** Notebook 06 saved its answers in `output/test_predictions.npz`, and this
notebook only reads those. Run it after 01 to 07.

**In:** every file in `data/` and `output/` **Out:** `output/validation.csv`

## 1. Load what the other notebooks saved

**Why this step:** Every test is answered from these files. If 07 wasn't run, its tests say "Not run" instead of failing.

**The problem:** Load the raw file, the split, the features, the saved answers, and set up the results table.

**How:** `record(no, what, result, verdict)` is my small function that adds one row to the table and prints it. `read_csv(path)` reads a .csv into a list of rows with `csv.DictReader`. `os.path.exists` is True when a file is there.\
`wafer.check_one_run()` stops the notebook when the saved files don't come from one run: if one notebook was run again by hand, its file is newer than the files made after it, and the tables would mix two runs.

In [1]:
import csv
import os
import time

import numpy as np
import joblib
from sklearn.base import clone

import wafer

# Stop if the saved files come from different runs
wafer.check_one_run()

original = np.load("../data/Wafer_Map_Datasets.npz")
split = np.load("output/split.npz")
features = np.load("output/features.npz", allow_pickle=True)
answers = np.load("output/test_predictions.npz")

maps = split["maps"]
labels = split["labels"]
train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]
y_test = answers["y_test"]
y_pred = answers["y_pred"]
y_base = answers["y_base"]

have_generator = os.path.exists("output/generated.npz") and os.path.exists("output/generated_scores.csv")
print("Generator results there:", have_generator)

rows = []    # one row per test


def record(no, what, result, verdict):
    rows.append({"No": no, "What": what, "Result": result, "Pass/Fail": verdict})
    print(no, verdict, "|", result)


def read_csv(path):
    with open(path, encoding="utf-8") as file:
        return list(csv.DictReader(file))

Generator results there: True


**What you should see:** "Generator results there: True" if notebook 07 has run.

## 2. Validation: does it do what the requirements ask?

Each cell below is one requirement, with its steps as comments. A requirement passes only
when every step holds.

In [2]:
# T1  FR-01  An 8 position label for every wafer
# Steps: predict on the test set. The shape is (wafers, 8), and only 0 and 1.
right_shape = y_pred.shape == (len(test_idx), 8)
only_0_1 = set(np.unique(y_pred)) <= {0, 1}

if right_shape and only_0_1:
    verdict = "Pass"
else:
    verdict = "Fail"
record("T1", "FR-01 8 position label for every wafer", f"shape {y_pred.shape}, values {np.unique(y_pred)}", verdict)

T1 Pass | shape (5535, 8), values [0 1]


T2 also counts something worth knowing: a forest with 8 separate answers can say yes to any mix of defects, also a mix that isn't one of the 38 types in the dataset.

In [3]:
# T2  FR-02  The label as a name, or Normal
# Steps: label_name() on 10 random predictions gives a name. All zeros gives Normal.
#        Count predictions that are a combination outside the 38 types.
rng = np.random.default_rng(1)
names = []
for i in rng.choice(len(y_pred), 10, replace=False):
    names.append(wafer.label_name(y_pred[i]))
    print("  ", y_pred[i], "->", names[-1])

normal_ok = wafer.label_name(np.zeros(8, dtype=int)) == "Normal"

known_types = []
for t in np.unique(labels, axis=0):
    known_types.append(tuple(t))

outside = 0
for p in y_pred:
    if tuple(p) not in known_types:
        outside += 1

if all(names) and normal_ok:
    verdict = "Pass"
else:
    verdict = "Fail"
record("T2", "FR-02 label becomes a name, or Normal",
       f"10 of 10 named, all zeros gives Normal: {normal_ok}, {outside} of {len(y_pred)} outside the 38 types", verdict)

   [1 0 0 0 0 0 0 0] -> Center
   [0 1 1 0 1 0 0 0] -> Donut + Edge-Loc + Loc
   [0 0 0 0 1 0 1 0] -> Loc + Scratch
   [1 0 1 0 0 0 1 0] -> Center + Edge-Loc + Scratch
   [0 0 0 1 0 0 1 0] -> Edge-Ring + Scratch
   [0 1 0 1 1 0 1 0] -> Donut + Edge-Ring + Loc + Scratch
   [1 0 1 0 1 0 1 0] -> Center + Edge-Loc + Loc + Scratch
   [0 0 1 0 1 0 0 0] -> Edge-Loc + Loc
   [0 0 0 1 0 0 1 0] -> Edge-Ring + Scratch
   [1 0 0 0 1 0 1 0] -> Center + Loc + Scratch
T2 Pass | 10 of 10 named, all zeros gives Normal: True, 0 of 5535 outside the 38 types


In [4]:
# T3  FR-03  Copies removed before the split
# Steps: count the different maps before and after cleaning. After: different equals rows.
#        Cleaning comes first: split.npz holds the cleaned maps, and the split points into them.
raw = original["arr_0"]
before = len(np.unique(raw.reshape(len(raw), -1), axis=0))
after = len(np.unique(maps.reshape(len(maps), -1), axis=0))

if after == len(maps):
    verdict = "Pass"
else:
    verdict = "Fail"
record("T3", "FR-03 copies removed before the split",
       f"{len(raw):,} loaded, {before:,} different, after cleaning {len(maps):,} rows and {after:,} different", verdict)

T3 Pass | 38,015 loaded, 36,991 different, after cleaning 36,890 rows and 36,890 different


In [5]:
# T4  FR-04  Value 3 handled in one way, with the reason written down
# Steps: count wafers with a 3 before. After: no value other than 0, 1, 2.
#        The reason is in notebook 02.
with_3 = (raw == 3).any(axis=(1, 2)).sum()
odd_left = (~np.isin(maps, [0, 1, 2])).sum()     # ~ means "not"

if odd_left == 0:
    verdict = "Pass"
else:
    verdict = "Fail"
record("T4", "FR-04 value 3 handled one way", f"{with_3} wafers with a 3 before, {odd_left} odd squares after", verdict)

T4 Pass | 105 wafers with a 3 before, 0 odd squares after


In [6]:
# T5  FR-05  All 38 types in every part
# Steps: count wafers per type per part. None is 0, Near-Full is in test, about 70/15/15.
types = np.unique(labels, axis=0)
missing = 0
for t in types:
    for part in [train_idx, val_idx, test_idx]:
        if (labels[part] == t).all(axis=1).sum() == 0:
            missing += 1

near_full = np.zeros(8, dtype=labels.dtype)
near_full[wafer.DEFECTS.index("Near-Full")] = 1
near_full_in_test = (labels[test_idx] == near_full).all(axis=1).sum()

total = len(train_idx) + len(val_idx) + len(test_idx)
shares = f"{len(train_idx) / total * 100:.1f}/{len(val_idx) / total * 100:.1f}/{len(test_idx) / total * 100:.1f}"

if missing == 0 and near_full_in_test > 0:
    verdict = "Pass"
else:
    verdict = "Fail"
record("T5", "FR-05 every type in every part",
       f"{len(types)} types, {missing} missing, Near-Full {near_full_in_test} in test, split {shares}", verdict)

T5 Pass | 38 types, 0 missing, Near-Full 21 in test, split 70.0/15.0/15.0


In [7]:
# T6  FR-06  A baseline next to the model
# Steps: the baseline and the guesses are scored in 04, the forest next to the baseline in 06, all in output/.
baseline_file = os.path.exists("output/baseline_validation.csv")
test_scores = read_csv("output/test_scores.csv")
has_both = "baseline_f1" in test_scores[0] and "f1" in test_scores[0]

if baseline_file and has_both:
    verdict = "Pass"
else:
    verdict = "Fail"
record("T6", "FR-06 baseline next to the model",
       f"baseline_validation.csv there: {baseline_file}, model and baseline in test_scores.csv: {has_both}", verdict)

T6 Pass | baseline_validation.csv there: True, model and baseline in test_scores.csv: True


In [8]:
# T7  FR-07  Scores per defect and per number of defects
# Steps: the groups of 1 to 4 defects all have wafers, and every defect has a score.
per_number = read_csv("output/test_per_number_of_defects.csv")
groups = {}
for row in per_number:
    groups[row["defects_on_wafer"]] = int(row["wafers"])

have_1_to_4 = True
for n in ["1", "2", "3", "4"]:
    if groups.get(n, 0) == 0:
        have_1_to_4 = False

if have_1_to_4 and len(test_scores) == 8:
    verdict = "Pass"
else:
    verdict = "Fail"
record("T7", "FR-07 per defect and per number of defects", f"wafers per group: {groups}", verdict)

T7 Pass | wafers per group: {'0': 150, '1': 1035, '2': 1950, '3': 1800, '4': 600, 'all': 5535}


In [9]:
# T8  FR-08  The 8 defect types ranked
# Steps: sorted by recall, lowest first, all 8 in it, none twice.
ranked = []
recalls = []
for row in test_scores:
    ranked.append(row["defect"])
    recalls.append(float(row["recall"]))

in_order = True
for k in range(len(recalls) - 1):
    if recalls[k] > recalls[k + 1]:
        in_order = False

if sorted(ranked) == sorted(wafer.DEFECTS) and in_order:
    verdict = "Pass"
else:
    verdict = "Fail"
record("T8", "FR-08 the 8 types ranked", "most to least problematic: " + ", ".join(ranked), verdict)

T8 Pass | most to least problematic: Random, Edge-Loc, Scratch, Loc, Donut, Center, Edge-Ring, Near-Full


In [10]:
# T9  FR-09  A generator for the 38 types, saved with label and settings (Should)
# Steps: all 38 types, every map with a label and its settings.
if have_generator:
    gen = np.load("output/generated.npz")
    gen_types = len(np.unique(gen["labels"], axis=0))
    complete = gen["labels"].shape[1] == 8 and len(gen["settings"]) == len(gen["maps"])
    if gen_types == len(types) and complete:
        verdict = "Pass"
    else:
        verdict = "Fail"
    record("T9", "FR-09 generator for the 38 types",
           f"{len(gen['maps']):,} maps, {gen_types} types, label and settings with each: {complete}", verdict)
else:
    record("T9", "FR-09 generator for the 38 types", "notebook 07 wasn't run", "Not run")

T9 Pass | 1,900 maps, 38 types, label and settings with each: True


In [11]:
# T10  FR-10  Where the misses sit (Could)
# Steps: the average map of the missed wafers per defect. A picture, so it is judged by eye in notebook 06 step 8.
record("T10", "FR-10 where the misses sit", "drawn in notebook 06 step 8", "Check by eye")

T10 Check by eye | drawn in notebook 06 step 8


## 3. Verification: does it work correctly?

These check the work itself. V1 and V2 are the data leakage risks. V4 trains the
model twice to prove the same seed gives the same answers.\
`clone(saved_forest)` makes a fresh, untrained forest with exactly the same settings as the saved one, so I don't
have to type the settings over. `np.isin(maps, [0, 1, 2])` is True for every value in the list.

In [12]:
# V1  No map in two parts: put two parts in one pile and count the different maps
flat = maps.reshape(len(maps), -1)
in_common = []
for a, b in [(train_idx, val_idx), (train_idx, test_idx), (val_idx, test_idx)]:
    pile = np.concatenate([flat[a], flat[b]])
    in_common.append(len(pile) - len(np.unique(pile, axis=0)))

if sum(in_common) == 0:
    verdict = "Pass"
else:
    verdict = "Fail"
record("V1", "no map in two parts", f"maps in common (train-val, train-test, val-test): {in_common}", verdict)

V1 Pass | maps in common (train-val, train-test, val-test): [0, 0, 0]


In [13]:
# V2  Every type in every part: the smallest count per part may not be 0
smallest = []
for part in [train_idx, val_idx, test_idx]:
    counts = []
    for t in types:
        counts.append(int((labels[part] == t).all(axis=1).sum()))
    smallest.append(min(counts))

if min(smallest) > 0:
    verdict = "Pass"
else:
    verdict = "Fail"
record("V2", "every type in every part", f"smallest type per part (train, val, test): {smallest}", verdict)

V2 Pass | smallest type per part (train, val, test): [93, 20, 21]


In [14]:
# V3  No value 3 left in the data the model sees
values = []
for v in np.unique(maps):
    values.append(int(v))

if values == [0, 1, 2]:
    verdict = "Pass"
else:
    verdict = "Fail"
record("V3", "no value 3 left", f"values in the cleaned maps: {values}", verdict)

V3 Pass | values in the cleaned maps: [0, 1, 2]


In [15]:
# V4  Same seed, same split and same scores (also QC-06)
# The split: make it twice with seed 42 and compare with the saved one.
first = wafer.make_split(labels, 42)
second = wafer.make_split(labels, 42)
same_split = True
for a, b, saved in zip(first, second, [train_idx, val_idx, test_idx]):
    if not (np.array_equal(a, b) and np.array_equal(a, saved)):
        same_split = False

# The scores: train two fresh forests with the saved settings and compare their validation answers
saved_forest = joblib.load("output/forest.joblib")
X_train = features["X_train"]
y_train = features["y_train"]
X_val = features["X_val"]

start = time.perf_counter()
run_1 = clone(saved_forest).fit(X_train, y_train).predict(X_val)
run_2 = clone(saved_forest).fit(X_train, y_train).predict(X_val)
same_scores = np.array_equal(run_1, run_2)
print(f"   (two forests trained in {time.perf_counter() - start:.0f} s)")

if same_split and same_scores:
    verdict = "Pass"
else:
    verdict = "Fail"
record("V4", "same seed, same split and scores",
       f"split the same: {same_split}, two trainings give the same answers: {same_scores}", verdict)

   (two forests trained in 6 s)
V4 Pass | split the same: True, two trainings give the same answers: True


## 4. Quality criteria

For QC-01 and QC-03 the measured value is the **lowest** defect and its number, not an average. The criterion is
per defect, and an average would hide the one that fails.

In [16]:
recall, precision = wafer.scores_per_defect(y_test, y_pred)
base_recall, base_precision = wafer.scores_per_defect(y_test, y_base)
all_right = wafer.share_all_eight_right(y_test, y_pred)
base_all_right = wafer.share_all_eight_right(y_test, y_base)

# QC-01: recall at least 70% on every defect
worst = np.argmin(recall)
if recall.min() >= 0.70:
    verdict = "Pass"
else:
    verdict = "Fail"
record("QC-01", "recall per type, min 70%", f"lowest {wafer.DEFECTS[worst]} {recall[worst] * 100:.1f}%", verdict)

# QC-02: all 8 right on at least 60% of the test wafers
if all_right >= 0.60:
    verdict = "Pass"
else:
    verdict = "Fail"
record("QC-02", "all 8 right, min 60%", f"{all_right * 100:.1f}% of test wafers", verdict)

# QC-03: precision at least 60% on every defect
worst_p = np.argmin(precision)
if precision.min() >= 0.60:
    verdict = "Pass"
else:
    verdict = "Fail"
record("QC-03", "precision, min 60%", f"lowest {wafer.DEFECTS[worst_p]} {precision[worst_p] * 100:.1f}%", verdict)

# QC-04: a higher F1 than the simple model on every defect, and more wafers all 8 right
f1 = wafer.f1_per_defect(recall, precision)
base_f1 = wafer.f1_per_defect(base_recall, base_precision)
better = (f1 > base_f1).sum()
closest = np.argmin(f1 - base_f1)
if better == 8 and all_right > base_all_right:
    verdict = "Pass"
else:
    verdict = "Fail"
record("QC-04", "beats the simple model on every type",
       f"F1 higher on {better} of 8, closest {wafer.DEFECTS[closest]} {f1[closest] * 100:.1f}% vs "
       f"{base_f1[closest] * 100:.1f}%, all 8 right {all_right * 100:.1f}% vs {base_all_right * 100:.1f}%", verdict)

QC-01 Pass | lowest Random 71.1%
QC-02 Pass | 68.3% of test wafers
QC-03 Pass | lowest Scratch 76.9%
QC-04 Fail | F1 higher on 7 of 8, closest Near-Full 93.3% vs 95.2%, all 8 right 68.3% vs 31.2%


In [17]:
# QC-05: on my own wafers at most 15 points lower
if have_generator:
    biggest_gap = None
    biggest_defect = ""
    for row in read_csv("output/generated_scores.csv"):
        gap = float(row["gap_points"])
        if biggest_gap is None or gap > biggest_gap:
            biggest_gap = gap
            biggest_defect = row["defect"]
    if biggest_gap <= 15:
        verdict = "Pass"
    else:
        verdict = "Fail"
    record("QC-05", "own wafers at most 15 points lower", f"biggest gap {biggest_defect} {biggest_gap:.1f} points", verdict)
else:
    record("QC-05", "own wafers at most 15 points lower", "notebook 07 wasn't run", "Not tested")

# QC-06 is the same check as V4
for row in rows:
    if row["No"] == "V4":
        v4 = row
record("QC-06", "same seed, same split and scores", "same as V4", v4["Pass/Fail"])

# QC-07 to QC-09: the times notebook 05 measured
timings = {}
for row in read_csv("output/timings.csv"):
    timings[row["criterion"]] = float(row["measured"])

t7 = timings["QC-07 training (s)"]
t8 = timings["QC-08 one wafer (ms)"]
t9 = timings["QC-09 1000 wafers (s)"]
record("QC-07", "training under 60 min", f"{t7:.0f} s", "Pass" if t7 < 3600 else "Fail")
record("QC-08", "one wafer under 50 ms", f"{t8:.1f} ms, features included", "Pass" if t8 < 50 else "Fail")
record("QC-09", "1000 wafers under 10 s", f"{t9:.2f} s, features included", "Pass" if t9 < 10 else "Fail")

QC-05 Fail | biggest gap Scratch 71.4 points
QC-06 Pass | same as V4
QC-07 Pass | 2 s
QC-08 Pass | 8.6 ms, features included
QC-09 Pass | 0.20 s, features included


## 5. The tables

**Why this step:** One file with every verdict, not numbers typed over from the screen.

**The problem:** Print the three tables, and save them as a csv.

**How:** The same rows, printed as markdown tables.

In [18]:
for title, start in [("Validation", "T"), ("Verification", "V"), ("Quality criteria", "QC")]:
    print(title)
    print("| No | Result | Pass/Fail |")
    print("|---|---|---|")
    for row in rows:
        if row["No"].startswith(start):
            print(f"| {row['No']} | {row['Result']} | {row['Pass/Fail']} |")
    print()

with open("output/validation.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["No", "What", "Result", "Pass/Fail"])
    for row in rows:
        writer.writerow([row["No"], row["What"], row["Result"], row["Pass/Fail"]])

passed = 0
for row in rows:
    if row["Pass/Fail"] == "Pass":
        passed += 1
print(f"Saved output/validation.csv, {passed} of {len(rows)} pass")

Validation
| No | Result | Pass/Fail |
|---|---|---|
| T1 | shape (5535, 8), values [0 1] | Pass |
| T2 | 10 of 10 named, all zeros gives Normal: True, 0 of 5535 outside the 38 types | Pass |
| T3 | 38,015 loaded, 36,991 different, after cleaning 36,890 rows and 36,890 different | Pass |
| T4 | 105 wafers with a 3 before, 0 odd squares after | Pass |
| T5 | 38 types, 0 missing, Near-Full 21 in test, split 70.0/15.0/15.0 | Pass |
| T6 | baseline_validation.csv there: True, model and baseline in test_scores.csv: True | Pass |
| T7 | wafers per group: {'0': 150, '1': 1035, '2': 1950, '3': 1800, '4': 600, 'all': 5535} | Pass |
| T8 | most to least problematic: Random, Edge-Loc, Scratch, Loc, Donut, Center, Edge-Ring, Near-Full | Pass |
| T9 | 1,900 maps, 38 types, label and settings with each: True | Pass |
| T10 | drawn in notebook 06 step 8 | Check by eye |

Verification
| No | Result | Pass/Fail |
|---|---|---|
| V1 | maps in common (train-val, train-test, val-test): [0, 0, 0] | Pass |


**What you should see:** Three tables, every row with Pass, Fail, Not run or Check by eye. A fail stays in the
table, with what it would take to fix it. That's more honest than a table where everything passes.

## Functions in this notebook

| Function | What it does | Where to read more |
|---|---|---|
| `sklearn.base.clone` | a fresh, untrained copy of a model with the same settings | [docs](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html) |
| `np.isin` | True for every value that's in a list | [docs](https://numpy.org/doc/stable/reference/generated/numpy.isin.html) |
| `np.concatenate` | puts arrays after each other | [docs](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html) |
| `np.array_equal` | True if two arrays are exactly the same | [docs](https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html) |
| `csv.DictReader` | reads a .csv row by row as name to value | [docs](https://docs.python.org/3/library/csv.html#csv.DictReader) |
| `os.path.exists` | True when a file is there | [docs](https://docs.python.org/3/library/os.path.html#os.path.exists) |